Capítulo HPC
==========


## Executando meu primeiro código em um Supercomputador!


Neste capítulo você vai aprender a diferença entre **prototipar** um código na sua máquina e **executá-lo de verdade** em um cluster, a Heisenberg.

A ideia central é:

- **Na sua máquina**, você usa este Jupyter Notebook (`.ipynb`) para escrever, testar e cronometrar pequenos trechos de código de forma interativa.
- **No cluster**, você **não** roda Jupyter. Em vez disso, salva o código pronto em um arquivo `.py` e o submete ao escalonador SLURM com o comando `sbatch`.

Por que não rodar Jupyter na Heisenberg? Porque um notebook fica *segurando* os recursos alocados o tempo todo, mesmo quando você está apenas pensando ou lendo. Em um cluster compartilhado isso desperdiça núcleos e memória que outras pessoas poderiam estar usando. O jeito correto é: o job entra na fila, roda o `.py`, grava o resultado em um arquivo e libera o nó.

### Questão [A]


Vamos começar com a biblioteca `socket`, que nos dá o *hostname* da máquina. Rode a célula abaixo **na sua máquina**:

In [ ]:
import socket

print(socket.gethostname())

Esse código imprime o nome da máquina onde ele está rodando. Quando você colocar esse mesmo trecho dentro de um `.py` na Heisenberg, o nome impresso será o do **nó de trabalho** onde o job caiu (por exemplo, `work3`), provando que ele rodou no cluster, e não no seu computador.

### Questão [B]


Agora um código que encontra os números primos de uma faixa. Rode na sua máquina e **anote o tempo**:

In [ ]:
import time

def encontrarprimos(start, end):
    listadeprimos = set()
    for i in range(start, end + 1):
        notprime = 0
        for j in range(start + 1, int(end / 2)):
            if i % j == 0 and i != j:
                notprime = 1
                break
        if notprime == 0:
            listadeprimos.add(i)
    return listadeprimos

tic = time.perf_counter()
print(sorted(encontrarprimos(1, 30000)))
toc = time.perf_counter()
print(f"O código rodou em {toc - tic:0.4f} segundos")

Você espera que este código rode mais rápido ou mais devagar no cluster do que no seu laptop? Mais adiante vamos testar isso de verdade — guarde a sua hipótese.

### Questão [C]


Vamos inverter uma matriz grande com `numpy`. Rode na sua máquina:

In [ ]:
import numpy as np
import time

a = np.random.random((4000, 4000))

tic = time.perf_counter()
np.linalg.inv(a)
toc = time.perf_counter()
print(f"O código rodou em {toc - tic:0.4f} segundos")

Se aparecer o erro `No module named 'numpy'`, instale a biblioteca no seu terminal (PowerShell/terminal) com:

```
pip install numpy
```

Depois rode a célula novamente.

### Questão [D]


Compare: a inversão foi mais rápida ou mais lenta no seu laptop?

No seu laptop, o `numpy` usa bibliotecas matemáticas otimizadas e paralelas e tende a usar **todos** os núcleos disponíveis. No cluster, o job recebe apenas os núcleos que você pedir no `job.sh` — então a quantidade de núcleos alocados muda diretamente o desempenho. É exatamente isso que vamos explorar a seguir.

## Rodando de verdade na Heisenberg (arquivos .py)


A partir daqui o código **sai do notebook** e vira um arquivo `.py`, que o cluster executa em lote (*batch*).

Você vai trabalhar com dois arquivos dentro da pasta `work` (acesse pelo WinSCP):

- `inverte_matriz.py` — o programa Python que será executado.
- `job.sh` — o script que diz ao SLURM **como** e **com quais recursos** rodar o programa.

### Questão [E]

Crie (ou edite) o arquivo `inverte_matriz.py` na pasta `work` com o conteúdo abaixo. É praticamente o mesmo código da Questão [C], mas em formato de script:

```python
import os
import time
import numpy as np

print("Rodando no nó:", os.uname()[1])
print("Núcleos para o numpy (OMP_NUM_THREADS):", os.environ.get("OMP_NUM_THREADS", "não definido"))

a = np.random.random((4000, 4000))

tic = time.perf_counter()
np.linalg.inv(a)
toc = time.perf_counter()
print(f"O código rodou em {toc - tic:0.4f} segundos")
```

### Questão [F]

Agora confira o arquivo `job.sh`. Ele deve estar assim:

```bash
#!/bin/bash
#SBATCH -n 1
#SBATCH --ntasks-per-node=1
#SBATCH -p batch-AMD
#SBATCH --mem=2000mb
#SBATCH --time=00:30:00
#SBATCH --job-name=inverte
#SBATCH --output=resultado-%j.out

source ~/.bashrc
conda activate ilumpy

# Faz o numpy usar o mesmo número de núcleos que pedimos ao SLURM
export OMP_NUM_THREADS=$SLURM_NTASKS
export OPENBLAS_NUM_THREADS=$SLURM_NTASKS

python inverte_matriz.py
```

O que cada linha faz:

- `#SBATCH -n 1` → pede **1** núcleo (tarefa) para o job.
- `--ntasks-per-node=1` → coloca essa tarefa em um único nó.
- `-p batch-AMD` → escolhe a partição (fila) de nós AMD.
- `--mem=2000mb` → reserva 2 GB de memória.
- `--time=00:30:00` → tempo máximo de 30 minutos; passou disso, o SLURM encerra o job.
- `--output=resultado-%j.out` → grava a saída em um arquivo, onde `%j` vira o número do job.
- as linhas `export ...` garantem que o `numpy` use a mesma quantidade de núcleos pedida.

Submeta o job com **1 núcleo** no terminal:

```
sbatch job.sh
```

Acompanhe a fila com:

```
squeue -u seu_usuário
```

Você verá algo como:

```
JOBID PARTITION    NAME   USER ST  TIME NODES NODELIST(REASON)
 531  batch-AMD inverte  james  R  0:05     1 work2
```

Quando o job terminar, abra o arquivo `resultado-531.out` (com o número do **seu** job) e anote o tempo de execução.

### Questão [G]

Vamos pedir **3 núcleos** e repetir. Edite o `job.sh` e troque as duas primeiras diretivas `#SBATCH`.

De:

```bash
#SBATCH -n 1
#SBATCH --ntasks-per-node=1
```

Para:

```bash
#SBATCH -n 3
#SBATCH --ntasks-per-node=3
```

Se ainda houver um job seu rodando, cancele-o primeiro. Descubra o `JOBID` com `squeue -u seu_usuário` e rode:

```
scancel JOBID
```

(troque `JOBID` pelo número do seu job, por exemplo `scancel 531`).

Submeta novamente:

```
sbatch job.sh
```

Abra o novo arquivo `resultado-<novo_job>.out` e compare o tempo com o da Questão [F].

**Comparação:** o tempo com 3 núcleos foi menor que com 1 núcleo? Por quê?

A inversão de matriz do `numpy` é paralelizável: ela distribui as contas entre os núcleos disponíveis, então mais núcleos costumam reduzir o tempo. Já o código de primos da Questão [B] é Python "puro" e roda em um único núcleo — dar mais núcleos a ele **não** o deixaria mais rápido. Entender essa diferença é o coração da computação de alto desempenho.

## Exercícios de prática


Os exercícios a seguir são apenas para você praticar antes da próxima aula. Use as células de código abaixo para experimentar à vontade — erre, teste e tente de novo.

### Parte 1 — Conceitos

Responda com suas próprias palavras (pode usar uma célula de texto ou só pensar):

1. Por que **não** devemos rodar o Jupyter direto em um nó do cluster?
2. Qual a diferença entre o **nó de login** (work0) e os **nós de trabalho** (work1–work8)?
3. Explique o que faz cada diretiva: `-n`, `--ntasks-per-node`, `-p`, `--mem`, `--time`, `--output`.
4. Na saída do `squeue`, o que significam as colunas `ST`, `NODES` e `NODELIST`? O que indica o estado `R`? E o estado `PD`?
5. Por que dar mais núcleos acelera a inversão de matriz, mas **não** acelera o código de primos?
6. Olhando o diagrama do cluster: se o seu job precisa de **1 GPU RTX 4090**, em quais nós ele poderia rodar? E se precisar de **2 GPUs RTX 4090**?
7. O que acontece se o job ultrapassar o tempo do `--time`? E se ultrapassar a memória do `--mem`?
8. Se você pedir mais memória do que existe em qualquer nó da partição, o que acontece com o seu job na fila?

### Parte 2 — Comandos SLURM

Treine direto no terminal da Heisenberg:

9. Liste todos os **seus** jobs na fila.
10. Submeta o `job.sh` e, logo em seguida, cancele-o.
11. Pesquise o comando `sinfo` e descubra quais partições existem e quantos nós estão livres.
12. Use o `squeue` para ver quantos jobs (de todos os usuários) estão na fila no momento.
13. Descubra como pedir um job com `--time=00:05:00` e veja se ele entra mais rápido na fila.

### Parte 3 — Programação e desempenho

Use as células de código abaixo:

14. Modifique o código de primos para apenas **contar** quantos primos existem entre 1 e 50000, e cronometre.
15. Melhore o algoritmo de primos: o laço interno vai até `end/2`, mas basta ir até `sqrt(i)`. Reescreva e compare os tempos.
16. Troque a inversão por uma **multiplicação** de matrizes `a @ b` (10000×10000) e cronometre.
17. Rode a inversão para os tamanhos 1000, 2000, 4000 e 8000 e monte uma tabela tempo × tamanho. O tempo cresce de forma linear?
18. Escreva um `.py` que imprima o hostname (`os.uname()[1]`) e o número de núcleos visíveis (`os.cpu_count()`), e imagine como ficaria o `job.sh` para rodá-lo.
19. (Desafio) Crie uma versão do código de primos usando o módulo `multiprocessing` para dividir a faixa entre vários processos. Agora mais núcleos ajudam?

In [ ]:
# Exercício 14 — conte os primos entre 1 e 50000 e cronometre
# Dica: reaproveite a função encontrarprimos da Questão [B]



In [ ]:
# Exercício 15 — versão otimizada com sqrt(i)
import math
import time

def primos_rapido(start, end):
    primos = set()
    # TODO: implemente usando range(2, int(math.sqrt(i)) + 1)
    return primos



In [ ]:
# Exercício 16 — multiplicação de matrizes
import numpy as np
import time

# TODO: crie a e b, multiplique com a @ b e cronometre



In [ ]:
# Exercício 17 — tempo da inversão para vários tamanhos
import numpy as np
import time

for n in [1000, 2000, 4000, 8000]:
    a = np.random.random((n, n))
    tic = time.perf_counter()
    np.linalg.inv(a)
    toc = time.perf_counter()
    print(f"n={n}: {toc - tic:0.4f} s")


## Playground


Espaço livre para você testar o que quiser.